# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² dataset using the `mlcroissant` library, referencing all schema entities by their `@id` fields.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` is installed in your environment
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR² dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant JSON-LD schema URL (dataset definition)
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load dataset and metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(metadata)

## 2. Data Overview
Review available record sets, their field IDs, and field details as referenced by their `@id`s.

Let's print each record set's `@id`, its fields, and their corresponding `@id` values for reference.

In [ ]:
# List record sets and their fields by @id

record_sets = []
for record_set in metadata.record_sets:
    print(f"RecordSet @id: {record_set['@id']}")
    record_sets.append(record_set['@id'])
    print("  Fields:")
    for field in record_set.get('field', []):
        if isinstance(field, dict) and '@id' in field:
            print(f"    - {field['@id']}")
        else:
            print(f"    - {field}")
    print()

## 3. Data Extraction
Load data from one or more record sets into DataFrames for analysis. We reference all record sets and field columns explicitly via their `@id` fields, as listed above.

For this notebook, we'll demonstrate with the main protein abundance quantification table (the main RecordSet indicated by the schema), and further steps can be repeated for others.

In [ ]:
# Collect all record set @ids (from previous cell)
# For demonstration, select the main protein table. Please replace the @id below if needed based on previous cell.
# If you see only one id, use that. Let's assume it's 'https://api.app.sen.science/frontiers/7154140/quantab_table'

main_record_set_id = record_sets[0] if record_sets else None  # Use first record set if available
print(f"Extracting records from main record set: {main_record_set_id}")

dataframes = {}
if main_record_set_id:
    records = list(dataset.records(record_set=main_record_set_id))
    df = pd.DataFrame(records)
    dataframes[main_record_set_id] = df
    print("Columns in DataFrame:", df.columns.tolist())
    display(df.head())
else:
    print("No record sets found in the schema. Please check metadata.")

## 4. Exploratory Data Analysis (EDA)
Let's perform some basic data processing:
* Filter proteins by a numeric field (e.g., peptide counts or abundance)
* Normalize that field
* (Optionally) Group by a categorical field, such as experiment, if present

**Note:** Please replace the field IDs below with the actual field `@id` for a numeric column in your data (as printed in the DataFrame above), such as 'peptide_count' or 'abundance'.

In [ ]:
# Pick a numeric field @id for analysis. Replace these with field/column names from the output above.
# For example, for column 'peptide_count', the @id may look like 'https://api.app.sen.science/frontiers/7154140/peptide_count'.

# Example: numeric_field_id = 'peptide_count' (use @id if available in df.columns)
numeric_field_id = None
for col in dataframes[main_record_set_id].columns:
    if 'count' in col or 'abundance' in col or 'MW' in col or 'coverage' in col:
        numeric_field_id = col
        break

if numeric_field_id:
    threshold = dataframes[main_record_set_id][numeric_field_id].mean()
    filtered_df = dataframes[main_record_set_id][
        dataframes[main_record_set_id][numeric_field_id] > threshold
    ].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean())
        / filtered_df[numeric_field_id].std()
    )
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, norm_col]].head())

    # Try grouping by a categorical attribute (if any field contains 'sample' or 'experiment', use it)
    group_field = None
    for col in filtered_df.columns:
        if 'sample' in col or 'experiment' in col or 'condition' in col:
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field}:")
        display(grouped_df.head())
else:
    print("No numeric field found automatically. Please set 'numeric_field_id' to a valid column name.")

## 5. Visualization
Visualize the distribution of the selected numeric field.

We'll plot a histogram for the selected numeric field and a boxplot grouped by a categorical field (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    plt.figure(figsize=(7,4))
    sns.histplot(data=dataframes[main_record_set_id], x=numeric_field_id, bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field:
        plt.figure(figsize=(10,4))
        sns.boxplot(data=filtered_df, x=group_field, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()


## 6. Conclusion
We loaded the FAIR² mass spectrometry dataset using `mlcroissant` directly from its Croissant schema, explored its record sets and fields by their `@id`s, performed basic filtering and normalization on a numeric field, and visualized protein distribution statistics. For full schema details and authoritative field mappings, always reference the Croissant JSON-LD document directly.